In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from model import ScalingLaw, SampleAlpha
from constants import lower_bounds, test_models, delete_models, Y_names_tidy, Y_names, B, lrs, scheduler_factors, reps, n_epochs, random_seed
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from sloth.sloth import Sloth

eps = 1e-3
Y_names = Y_names[0]

# To aggregate individual models
class JoinModels():
    def __init__(self, models):
        self.models = models
    
    def predict(self, X, D):
        Y_hat = np.hstack([model.predict(X, D) for model in self.models])
        return Y_hat
    
# Model fitting
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

Data

In [2]:
# Loading
df = pd.merge(pd.read_csv('data/df_full_v1.csv').drop('Unnamed: 0', axis=1),
              pd.read_csv('data/df_full_v2.csv').drop('Unnamed: 0', axis=1), 
              on=['model', 'family', 'size', 'tokens', 'flops'], how='outer')

# Creating data objects
Y = np.array(df.loc[:,Y_names])
Y = np.clip(Y, a_min=eps, a_max=1-eps)
        
X = np.log(np.array(df.loc[:,['size','tokens']]))
X = np.hstack((X,(X[:,0]*X[:,1])[:,None]))

F = np.array(df.loc[:,['size','tokens']])
F = np.log(F[:,0]*F[:,1]).reshape((-1,1))

D = np.array(pd.get_dummies(df.family)).astype(int)
I = np.ones(shape=(D.shape[0],1))
C = np.array([lower_bounds[s] for s in Y_names]).reshape((1,-1))

# Data split
test = []
for l in list(test_models.values()):
    test+=l
train = []
for l in list(delete_models.values()):
    train+=l

#0.084 -> 0.039
#train_idx = np.array([m not in delete_models[family] for m in df.model])
#test_idx = np.array([m in test_models[family] for m in df.model])
train_idx = np.array([m not in train for m in df.model])
test_idx = np.array([m in test for m in df.model])
test_models_list = list(df.model[test_idx])

X_train, F_train, Y_train, D_train, I_train = X[train_idx], F[train_idx], Y[train_idx], D[train_idx], I[train_idx]
X_test, F_test, Y_test, D_test, I_test = X[test_idx], F[test_idx], Y[test_idx], D[test_idx], I[test_idx]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
F_train = scaler.fit_transform(F_train)
F_test = scaler.transform(F_test)

Training

In [3]:
dims = [3, 4, 5]
models = {}
predictions = {}

In [ ]:
# Unique intercept + FLOPs (Owen)
print("**** Unique intercept + FLOPs (Owen) ****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(F_train, I_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['flops'] = JoinModels(ind_models)
predictions['flops'] = models['flops'].predict(F_test, I_test)

# Unique intercept + Size/Tokens/Interaction
print("**** Unique intercept + Size/Tokens/Interaction ****")
ind_models = []
for j in tqdm(range(len(Y_names))):
    ind_models.append(Sloth(d=1))
    ind_models[-1].fit(X_train, I_train, Y_train[:,j:(j+1)], C[:,j:(j+1)], train_link=False, fit_C=True, positive_w=False, verbose=False, device='cpu')
models['size-tokens-inter'] = JoinModels(ind_models)
predictions['size-tokens-inter'] = models['size-tokens-inter'].predict(X_test, I_test)

# Simple Sloth
print("**** Simple Sloth ****")
for dim in tqdm(dims, desc='dims'):
    models[f'simple-sloth_{dim}'] = Sloth(d=dim)
    models[f'simple-sloth_{dim}'].fit(X_train, D_train, Y_train, C, train_link=False, fit_C=False, positive_w=False, verbose=False, device='cpu')
    predictions[f'simple-sloth_{dim}'] = models[f'simple-sloth_{dim}'].predict(X_test, D_test)

# Sloth
print("**** Sloth ****")
for dim in tqdm(dims, desc='dims'):
    models[f'sloth_{dim}'] = Sloth(d=dim)
    models[f'sloth_{dim}'].fit(X_train, D_train, Y_train,
                               C0=C,
                               W1_X0=models[f'simple-sloth_{dim}'].W1_X.numpy(),
                               W1_D0=models[f'simple-sloth_{dim}'].W1_D.numpy(),
                               W20=models[f'simple-sloth_{dim}'].W2.numpy(),
                               b20=models[f'simple-sloth_{dim}'].b2.numpy(),
                               train_link=True, fit_C=True, positive_w=False, verbose=False, device='cpu')
    predictions[f'sloth_{dim}'] = models[f'sloth_{dim}'].predict(X_test, D_test)

In [ ]:
# Ours
print("**** Ours ****")
for dim in tqdm(dims, desc='dims'):
    models[f'ours_{dim}'] = ScalingLaw(dim)
    models[f'ours_{dim}'].fit(X_train, Y_train, D_train, C,
                            B = B,
                            lrs = lrs,
                            scheduler_factors = scheduler_factors,
                            reps = reps,
                            n_epochs = n_epochs,    
                            verbose = True,
                            device = device)
    models[f'ours_{dim}'].predict(X_train, Y_train, D_train, X_test, D_test, C)

In [ ]:
np.save(f"models/predictive_analysis/models.npy", models)
np.save(f"models/predictive_analysis/predictions.npy", predictions)

Results

In [ ]:
mask = ~np.isnan(Y_test)